# 03. 백테스트

목적: 전략 성과 평가

- 거래 시뮬레이션
- 누적 수익률, MDD, Sharpe Ratio 계산
- 성과 시각화
- 최종 검증

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime

import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 6)

## 백테스트 엔진

In [ ]:
def backtest_strategy(df, initial_cash=10000000):
    """간단한 백테스트
    
    가정:
    - 신호 = 1: 모든 현금으로 매수
    - 신호 = -1: 전량 매도
    - 거래비용 미포함
    """
    
    df = df.copy()
    df['Cash'] = initial_cash
    df['Position'] = 0  # 보유 주식 수
    df['Portfolio_Value'] = initial_cash
    
    cash = initial_cash
    position = 0
    
    for i in range(len(df)):
        current_price = df.iloc[i]['Close']
        signal = df.iloc[i]['Signal']
        
        # 매수 신호
        if signal == 1 and position == 0:  # 신호 발생 시 1주 매수
            qty = 1
            cost = current_price * qty
            if cash >= cost:
                position += qty
                cash -= cost
        
        # 매도 신호
        elif signal == -1 and position > 0:
            proceeds = current_price * position
            cash += proceeds
            position = 0
        
        # 포트폴리오 가치
        portfolio_value = cash + (position * current_price)
        
        df.iloc[i, df.columns.get_loc('Cash')] = cash
        df.iloc[i, df.columns.get_loc('Position')] = position
        df.iloc[i, df.columns.get_loc('Portfolio_Value')] = portfolio_value
    
    return df

# 데이터 로드
ticker = '005930'
df = pd.read_csv(f'../data/processed/{ticker}_1y.csv', index_col=0, parse_dates=True)

# 지표 계산 (02_strategy.ipynb에서 정의한 함수 사용)
df['MA20'] = df['Close'].rolling(window=20).mean()
delta = df['Close'].diff()
gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
rs = gain / loss
df['RSI'] = 100 - (100 / (1 + rs))
df['Momentum'] = (df['Close'] / df['Close'].shift(10) - 1) * 100

# 신호 생성
df['Signal'] = 0
buy_signal = (df['Close'] > df['MA20'] * 1.01) & (df['Momentum'] > 2.0)
df.loc[buy_signal, 'Signal'] = 1
sell_signal = (df['Close'] < df['MA20'] * 0.98) | (df['RSI'] > 70)
df.loc[sell_signal, 'Signal'] = -1

# 백테스트 실행
df = backtest_strategy(df)

print("백테스트 완료")
print(df[['Close', 'MA20', 'Signal', 'Position', 'Cash', 'Portfolio_Value']].tail())

## 성과 지표 계산

In [ ]:
# 수익률 계산
initial_value = 10000000
final_value = df['Portfolio_Value'].iloc[-1]
total_return = (final_value - initial_value) / initial_value * 100

# 일 수익률
df['Daily_Return'] = df['Portfolio_Value'].pct_change()

# MDD (Maximum Drawdown)
cummax = df['Portfolio_Value'].expanding().max()
drawdown = (df['Portfolio_Value'] - cummax) / cummax
mdd = drawdown.min() * 100

# Sharpe Ratio (무위험율 0 가정)
annual_return = total_return
daily_volatility = df['Daily_Return'].std()
annual_volatility = daily_volatility * np.sqrt(252)
sharpe_ratio = annual_return / (annual_volatility * 100) if annual_volatility > 0 else 0

# 거래 통계
buy_signals = (df['Signal'] == 1).sum()
sell_signals = (df['Signal'] == -1).sum()

print("=" * 50)
print("백테스트 성과 요약")
print("=" * 50)
print(f"초기 자산: {initial_value:,.0f}원")
print(f"최종 자산: {final_value:,.0f}원")
print(f"총 수익률: {total_return:.2f}%")
print(f"최대 낙폭 (MDD): {mdd:.2f}%")
print(f"Sharpe Ratio: {sharpe_ratio:.3f}")
print(f"연 변동성: {annual_volatility * 100:.2f}%")
print()
print(f"매수 신호: {buy_signals}")
print(f"매도 신호: {sell_signals}")
print("=" * 50)

## 성과 시각화

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 포트폴리오 가치
ax = axes[0, 0]
ax.plot(df.index, df['Portfolio_Value'], label='Portfolio Value', linewidth=2)
ax.fill_between(df.index, initial_value, df['Portfolio_Value'], alpha=0.3)
ax.axhline(y=initial_value, color='r', linestyle='--', alpha=0.5)
ax.set_title('포트폴리오 가치')
ax.set_ylabel('Value (KRW)')
ax.legend()
ax.grid(True, alpha=0.3)

# 누적 수익률
ax = axes[0, 1]
cumulative_return = (df['Portfolio_Value'] / initial_value - 1) * 100
ax.plot(df.index, cumulative_return, linewidth=2)
ax.axhline(y=0, color='r', linestyle='--', alpha=0.5)
ax.fill_between(df.index, 0, cumulative_return, alpha=0.3)
ax.set_title('누적 수익률')
ax.set_ylabel('Return (%)')
ax.grid(True, alpha=0.3)

# 드로다운
ax = axes[1, 0]
ax.fill_between(df.index, 0, drawdown * 100, alpha=0.3, color='red')
ax.set_title('드로다운 (Drawdown)')
ax.set_ylabel('Drawdown (%)')
ax.grid(True, alpha=0.3)

# 일 수익률 분포
ax = axes[1, 1]
ax.hist(df['Daily_Return'].dropna() * 100, bins=50, alpha=0.7, edgecolor='black')
ax.set_title('일 수익률 분포')
ax.set_xlabel('Daily Return (%)')
ax.axvline(x=0, color='r', linestyle='--')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 결론

In [ ]:
print("\n" + "="*60)
print("전략 평가")
print("="*60)

if total_return > 0:
    print("✓ 양수 수익: 전략이 검증되었습니다.")
else:
    print("✗ 음수 수익: 추가 개선 필요합니다.")

if mdd > -20:
    print("✓ 낮은 MDD: 리스크 관리가 양호합니다.")
else:
    print("✗ 높은 MDD: 손실 관리 개선 필요합니다.")

if sharpe_ratio > 0.5:
    print("✓ 높은 Sharpe Ratio: 위험 대비 수익이 우수합니다.")
else:
    print("✗ 낮은 Sharpe Ratio: 수익성 개선 필요합니다.")

print("\n→ 위 분석을 바탕으로 파라미터를 조정하고")
print("  02_strategy.ipynb에서 신호 로직을 개선하세요.")
print("="*60)